# Vignette

`anndata-fcs` converts between FCS files and `AnnData` objects in both directions with consistent metadata:

In [1]:
import os
import tempfile

import flowio
import numpy as np
import pandas as pd

from anndata_fcs import read_fcs, write_fcs

DATA_DIR = os.path.join(os.getcwd(), "..", "..", "test")
FCS_FILENAME = os.path.join(DATA_DIR, "data", "HTS_BD_LSR_II_Mixed_Specimen_001_D6_D06.fcs")
OUT_DIR = tempfile.mkdtemp()

## Read FCS to AnnData

`read_fcs` reads the file with `flowio` and returns an `AnnData` object with events as observations and channels as
variables.

In [2]:
adata_fcs = read_fcs(FCS_FILENAME)
adata_fcs

AnnData object with n_obs × n_vars = 14945 × 11
    var: 'pnr', 'pne', 'pnv', 'png', 'pndisplay', 'pnbs', 'pnms'
    uns: 'fcs'
    layers: None (.X)

By default, `X` holds the event data.

In [3]:
adata_fcs.to_df().head()

,FSC-A,FSC-H,FSC-W,SSC-A,SSC-H,SSC-W,FITC-A,PerCP-Cy5-5-A,AmCyan-A,PE-TxRed YG-A,Time
0,-28531.250000,10.0,0.0,700.149963,1656.0,27708.351562,98.799995,54.149998,164.220001,120.360001,0.2
1,-49414.878906,8.0,0.0,1275.849976,2278.0,36705.050781,155.800003,13.300000,161.840012,94.860001,0.4
2,-58684.320312,14.0,0.0,-512.049988,472.0,0.000000,22.799999,8.550000,172.550003,85.680000,0.5
3,-3857.839844,432.0,0.0,276.449982,1339.0,13530.564453,-49.399998,34.200001,157.080002,89.759995,0.7
4,22825.830078,4606.0,262143.0,-505.399994,472.0,0.000000,90.250000,9.500000,330.820007,76.500000,0.7


Per-channel keywords are stored as `var` columns named `pn<suffix>`.

In [4]:
adata_fcs.var

,pnr,pne,pnv,png,pndisplay,pnbs,pnms
FSC-A,262144,"0,0",611,1.0,LOG,-1,0
FSC-H,262144,"0,0",611,1.0,LOG,-1,0
FSC-W,262144,"0,0",611,1.0,<NA>,-1,0
SSC-A,262144,"0,0",210,1.0,LOG,-1,0
SSC-H,262144,"0,0",210,1.0,LOG,-1,0
SSC-W,262144,"0,0",210,1.0,<NA>,-1,0
FITC-A,262144,"0,0",580,1.0,LOG,-1,0
PerCP-Cy5-5-A,262144,"0,0",580,1.0,LOG,-1,0
AmCyan-A,262144,"0,0",550,1.0,LOG,-1,0
PE-TxRed YG-A,262144,"0,0",500,1.0,LOG,-1,0


All global keywords are kept in `uns["fcs"]["text"]`.

In [5]:
text = adata_fcs.uns["fcs"]["text"]
pd.Series({key: text[key] for key in ["cyt", "creator", "date", "btim", "etim", "timestep", "spill"]})

cyt                                                     LSRII
creator                      BD FACSDiva Software Version 6.2
date                                              26-OCT-2012
btim                                                 18:08:10
etim                                                 18:08:21
timestep                                                 0.01
spill       4,FITC-A,PerCP-Cy5-5-A,AmCyan-A,PE-TxRed YG-A,...
dtype: str

Keywords describing the binary layout of the file (offsets, `$DATATYPE`, `$BYTEORD`, `$PnB`, ...) are kept for
reference in `uns["fcs"]["source"]`.

In [6]:
{key: value for key, value in adata_fcs.uns["fcs"].items() if key not in {"text", "analysis"}}

{'schema_version': 1,
 'version': '3.0',
 'preprocessed': False,
 'source': {'beginanalysis': '0',
  'endanalysis': '0',
  'beginstext': '0',
  'endstext': '0',
  'begindata': '2365',
  'enddata': '659944',
  'tot': '14945',
  'par': '11',
  'mode': 'L',
  'byteord': '4,3,2,1',
  'datatype': 'F',
  'nextdata': '0',
  'pnb': ['32', '32', '32', '32', '32', '32', '32', '32', '32', '32', '32']},
 'filename': 'HTS_BD_LSR_II_Mixed_Specimen_001_D6_D06.fcs'}

### Write AnnData to FCS

`write_fcs` writes event data and keywords.

In [7]:
fcs_copy = os.path.join(OUT_DIR, "lsr_ii_copy.fcs")
write_fcs(adata_fcs, fcs_copy, index_channel=None)

In [8]:
original = flowio.FlowData(FCS_FILENAME)
copy = flowio.FlowData(fcs_copy)

np.testing.assert_array_equal(copy.as_array(preprocess=False), original.as_array(preprocess=False))
assert copy.pnn_labels == original.pnn_labels
assert copy.pns_labels == original.pns_labels

Keywords describing the binary layout and the keywords marking the file as a copy differ.

Following the FCS 3.1 standard, `$ORIGINALITY` is set to `DataModified` and `$LAST_MODIFIED` to the time of writing.

In [9]:
keywords = pd.DataFrame({"original": pd.Series(original.text), "copy": pd.Series(copy.text)})
keywords[keywords["original"] != keywords["copy"]]

,original,copy
anndata_fcs_schema,NaN,1
begindata,2365,2419
byteord,"4,3,2,1","1,2,3,4"
enddata,659944,659998
last_modified,NaN,15-NOV-2026 20:23:12.06
originality,NaN,DataModified
tot,14945,14945


Reading the copy again gives the same AnnData object.

In [10]:
adata_copy = read_fcs(fcs_copy)

np.testing.assert_array_equal(adata_copy.X, adata_fcs.X)
pd.testing.assert_frame_equal(adata_copy.var[adata_fcs.var.columns], adata_fcs.var)

modified = {"originality", "last_modified"}
assert {k: v for k, v in adata_copy.uns["fcs"]["text"].items() if k not in modified} == adata_fcs.uns["fcs"]["text"]

### FCS Preprocessing Support

With `preprocess=True`, gain (`$PnG`), log (`$PnE`) and time (`$TIMESTEP`) scaling are applied to the event data.

In [11]:
adata_scaled = read_fcs(FCS_FILENAME, preprocess=True)
adata_scaled

AnnData object with n_obs × n_vars = 14945 × 11
    var: 'pnr', 'pne', 'pnv', 'png', 'pndisplay', 'pnbs', 'pnms'
    uns: 'fcs'
    layers: None (.X)

In [12]:
pd.DataFrame(
    {
        "raw": adata_fcs[:5, "Time"].X.ravel(),
        "preprocessed": adata_scaled[:5, "Time"].X.ravel(),
    }
)

,raw,preprocessed
0,0.2,0.002
1,0.4,0.004
2,0.5,0.005
3,0.7,0.007
4,0.7,0.007


In [13]:
pd.DataFrame({"raw": adata_fcs.var.loc["Time"], "preprocessed": adata_scaled.var.loc["Time"]})

,raw,preprocessed
pnr,262144,262144
pne,"0,0","0,0"
pnv,<NA>,<NA>
png,0.01,1.0
pndisplay,<NA>,<NA>
pnbs,0,0
pnms,0,0
